In [3]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

In [4]:
static_url = "https://en.wikipedia.org/w/index.php?title=List_of_Falcon_9_and_Falcon_Heavy_launches&oldid=1027686922"

In [5]:
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
}
response = requests.get(static_url, headers=headers)

In [6]:
response.status_code

200

In [7]:
soup = BeautifulSoup(response.text, "html.parser")

In [8]:
soup.title.text

'List of Falcon 9 and Falcon Heavy launches - Wikipedia'

In [9]:
html_tables = soup.find_all("table")

In [10]:
len(html_tables)

29

In [11]:
first_launch_table = html_tables[3]

In [12]:
headers = first_launch_table.find_all("th")

In [13]:
headers

[<th id="mwAms" scope="col">Flight No.</th>,
 <th id="mwAmw" scope="col">Date and<br id="mwAm0"/>time (<a href="https://en.wikipedia.org/wiki/Coordinated_Universal_Time" id="mwAm4" rel="mw:WikiLink" title="Coordinated Universal Time">UTC</a>)</th>,
 <th id="mwAm8" scope="col"><a href="https://en.wikipedia.org/wiki/List_of_Falcon_9_first-stage_boosters" id="mwAnA" rel="mw:WikiLink" title="List of Falcon 9 first-stage boosters">Version,<br id="mwAnE"/>Booster</a><sup about="#mwt214" class="mw-ref reference" data-mw='{"name":"ref","attrs":{"name":"booster","group":"lower-alpha"},"body":{"html":""},"parts":[{"template":{"target":{"wt":"efn","href":"./Template:Efn"},"params":{"name":{"wt":"booster"}},"i":0}}]}' id="cite_ref-booster_11-1" rel="dc:references" typeof="mw:Transclusion mw:Extension/ref"><a data-mw-group="lower-alpha" href="#cite_note-booster-11" id="mwAnI"><span class="mw-reflink-text" id="mwAnM"><span class="cite-bracket" id="mwAnQ">[</span>b<span class="cite-bracket" id="mwAnU

In [14]:
column_names = []

In [15]:
for th in headers:
    clean_name = th.text.strip()
    if clean_name is not None and len(clean_name) > 0:
        column_names.append(clean_name)

In [17]:
column_names

['Flight No.',
 'Date andtime (UTC)',
 'Version,Booster[b]',
 'Launch site',
 'Payload[c]',
 'Payload mass',
 'Orbit',
 'Customer',
 'Launchoutcome',
 'Boosterlanding',
 '8',
 '9',
 '10',
 '11',
 '12',
 '13']

In [18]:
launch_dict = {
    'Flight No.': [],
    'Date': [],
    'Time': [],
    'Version Booster': [],
    'Launch Site': [],
    'Payload': [],
    'Payload mass': [],
    'Orbit': [],
    'Customer': [],
    'Launch outcome': [],
    'Booster landing': []
}

In [19]:
len(launch_dict)

11

In [20]:
extracted_row = 0

for row in first_launch_table.find_all("tr"):
    th = row.find("th")
    if th and th.text.strip().isdigit():
        cells = row.find_all("td")
        launch_dict['Flight No.'].append(th.text.strip())
        date_time = list(cells[0].strings)
        launch_dict['Date'].append(date_time[0].strip(','))
        launch_dict['Time'].append(date_time[1].strip() if len(date_time) > 1 else "")
        launch_dict['Version Booster'].append(cells[1].text.strip())
        launch_dict['Launch Site'].append(cells[2].text.strip())
        launch_dict['Payload'].append(cells[3].text.strip())
        launch_dict['Payload mass'].append(cells[4].text.strip())
        launch_dict['Orbit'].append(cells[5].text.strip())
        launch_dict['Customer'].append(cells[6].text.strip())
        launch_dict['Launch outcome'].append(cells[7].text.strip())
        launch_dict['Booster landing'].append(cells[8].text.strip())
        extracted_row += 1

In [21]:
extracted_row

6

In [22]:
launch_dict

{'Flight No.': ['8', '9', '10', '11', '12', '13'],
 'Date': ['6 January 2014',
  '18 April 2014',
  '14 July 2014',
  '5 August 2014',
  '7 September 2014',
  '21 September 2014'],
 'Time': ['22:06', '19:25', '15:15', '08:00', '05:00', '05:52'],
 'Version Booster': ['F9 v1.1',
  'F9 v1.1',
  'F9 v1.1',
  'F9 v1.1',
  'F9 v1.1B1011[8]',
  'F9 v1.1B1010[8]'],
 'Launch Site': ['CCAFS,SLC-40',
  'Cape Canaveral,LC-40',
  'Cape Canaveral,LC-40',
  'Cape Canaveral,LC-40',
  'Cape Canaveral,LC-40',
  'Cape Canaveral,LC-40'],
 'Payload': ['Thaicom 6[22]',
  'SpaceX CRS-3[22](Dragon C105)',
  'Orbcomm-OG2-1(6 satellites)[22]',
  'AsiaSat 8[22][54][55]',
  'AsiaSat 6[22][54][58]',
  'SpaceX CRS-4[22](Dragon C106.1)'],
 'Payload mass': ['3,325\xa0kg (7,330\xa0lb)',
  '2,296\xa0kg (5,062\xa0lb)[45]',
  '1,316\xa0kg (2,901\xa0lb)',
  '4,535\xa0kg (9,998\xa0lb)',
  '4,428\xa0kg (9,762\xa0lb)',
  '2,216\xa0kg (4,885\xa0lb)[62]'],
 'Orbit': ['GTO', 'LEO (ISS)', 'LEO', 'GTO', 'GTO', 'LEO (ISS)'],
 'Cus

In [23]:
df = pd.DataFrame(launch_dict)

In [24]:
df.head()

,Flight No.,Date,Time,Version Booster,Launch Site,Payload,Payload mass,Orbit,Customer,Launch outcome,Booster landing
0,8,6 January 2014,22:06,F9 v1.1,"CCAFS,SLC-40",Thaicom 6[22],"3,325 kg (7,330 lb)",GTO,Thaicom,Success[42],No attempt[43]
1,9,18 April 2014,19:25,F9 v1.1,"Cape Canaveral,LC-40",SpaceX CRS-3[22](Dragon C105),"2,296 kg (5,062 lb)[45]",LEO (ISS),NASA (CRS),Success,Controlled(ocean) [d][46]
2,10,14 July 2014,15:15,F9 v1.1,"Cape Canaveral,LC-40",Orbcomm-OG2-1(6 satellites)[22],"1,316 kg (2,901 lb)",LEO,Orbcomm,Success[51],Controlled(ocean)[d][46]
3,11,5 August 2014,08:00,F9 v1.1,"Cape Canaveral,LC-40",AsiaSat 8[22][54][55],"4,535 kg (9,998 lb)",GTO,AsiaSat,Success[56],No attempt[57]
4,12,7 September 2014,05:00,F9 v1.1B1011[8],"Cape Canaveral,LC-40",AsiaSat 6[22][54][58],"4,428 kg (9,762 lb)",GTO,AsiaSat,Success[59],No attempt


In [25]:
df.to_csv("spacex_web_scraped.csv", index=False)